# Temporal LoRA Adapters

<a href="https://colab.research.google.com/github/Text-Machine/temporal-adapters/blob/train/temporal-adapters.ipynb" target="_parent\"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

This notebooks trains temporal adapters on ECCO data.

We train adapters per decade (windom) and move each window with five year (step).

In [6]:
# Import necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset
import json
import logging
import re
from datasets import Dataset
import pandas as pd
from pathlib import Path
from typing import Optional, Dict, Any

In [ ]:
# Download NLTK punkt tokenizer for sentence splitting
import nltk
nltk.download('punkt_tab')
nltk.download('punkt', quiet=True)  
from nltk.tokenize import sent_tokenize

print("✓ NLTK punkt downloaded")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/kasparbeelen/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✓ NLTK punkt downloaded


In [ ]:
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using CUDA GPU: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple MPS")
else:
    device = "cpu"
    print("Using CPU - you will need to use a GPU to train models")



Using Apple MPS


In [ ]:
# Authenticate with Hugging Face (optional, for private models)
from huggingface_hub import login
login()  # Uncomment if you need to access private models

In [3]:
# Uncomment the line below to download the data from Google Drive using gdown
#!gdown 11wfdV7j1TBv_i9XOiT8G8V4NxnJTxezz

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [5]:
def load_csv_as_dataset(csv_paths):
    """Load CSV files and convert to Hugging Face Dataset."""
    all_data = []
    
    for csv_path in csv_paths:
        logger.info(f"Loading {Path(csv_path).name}...")
        df = pd.read_csv(csv_path)
        logger.info(f"  Rows: {len(df)}, Columns: {list(df.columns)}")
        all_data.append(df)
    
    combined_df = pd.concat(all_data, ignore_index=True)
    logger.info(f"Total rows: {len(combined_df)}")
    
    dataset = Dataset.from_pandas(combined_df)
    return dataset


In [8]:
base_path = '../data-processing-code/data'
csv_path = Path(base_path).glob('*_cleaned.csv')
dataset = load_csv_as_dataset(csv_path)

INFO:__main__:Loading ecco_pages_cleaned.csv...
INFO:__main__:  Rows: 169051, Columns: ['author', 'place', 'date', 'page_text', 'converted_date']
INFO:__main__:Total rows: 169051


In [22]:
dataset[0]

{'author': 'Defoe, Daniel, 1661?-1731.',
 'place': 'London :',
 'date': 1706,
 'page_text': "D' FOE 's Answer to the Quakers  Catechism:  OR, A Dark  LANTHORN  for a Friend of the Light. To the READER. \n \n A Quaker with's dark Lanthorn light. \n Is here exposed to your sight; \n Stript off's nice Vizard and fair Paint, \n Wherein he us'd to Ape a Saint. \n So false Fires may delude our Eyes, \n And seem like Stars to guild the Skies; \n Till Reason proves they owe their Birth \n To stinking vapours of the Earth. \n This Hypocrite we here essay, \n In's proper Colours to display; \n Whose Yea and Nay in mischief goes \n Beyond the Hectors damning Oaths \n A Play-house Beau, is not so Gay, \n As now a Days the Yea and Nay: \n Whose Wigg in Curles, with Powder Drest, \n Makes him as Wicked as the rest; \n And seems to Act so very oddly, \n You'd Swear he's fallen from the Godly: \n For when he looks the most Precise, \n He tells you damn'd confounded Lyes. \n \n D' Foe, &c. London,  Pri

In [15]:
adapters_windows = [(year, year+10) for year in list(range(1700,1795,5))]
adapters_windows[:2]

[(1700, 1710), (1705, 1715)]

In [26]:
dataset_decade = dataset.filter(lambda x: adapters_windows[0][0] <= x['converted_date'] < adapters_windows[0][1])

Filter:   0%|          | 0/169051 [00:00<?, ? examples/s]

In [27]:


def text_preprocess_function(example):
    sentences = {'texts':[{"text": sent} for sent in sent_tokenize(example["page_text"])]}
    return sentences

dataset_decade_processed = dataset_decade.map(text_preprocess_function, batched=False,  remove_columns=dataset.column_names)

Map:   0%|          | 0/7764 [00:00<?, ? examples/s]

In [30]:
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/model.safetensors.index.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/model.safetensors.index.json "HTTP/1.1 200 OK"


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B/revision/main "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/tokenizer.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/tokenizer.json "HTTP/1.1 200 OK"


tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/tokenizer.model "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/special_tokens_map.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B "HTTP/1.1 200 OK"


In [32]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# 1) Configure LoRA
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# 2) Create trainer (example)
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(output_dir="lora-adapter", num_train_epochs=1, per_device_train_batch_size=2, packing=True),
    train_dataset=dataset_decade_processed,
    peft_config=peft_config,
)
trainer.train()

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`